In [ ]:
# DFU Phase-3 XAI — GPU Resume-Safe V3 FIXED loader
import urllib.request, base64, zlib, hashlib

VERSION = "DFU_PHASE3_XAI_GPU_RESUME_V3_FIXED_LOADER_20260811"
SOURCE_CONTAINER_COMMIT = "6bda1d466e1f472fa04559d76638402e1148514f"
EXPECTED_SOURCE_SHA256 = "9ba6e35e8e19c771fc7c6e8fbfcd27b7c2c882ff16878206f1e3660ffaa6057b"
SOURCE_CONTAINER_URL = (
    "https://raw.githubusercontent.com/AzizulHakim00/DFU-ImageGuard/"
    + SOURCE_CONTAINER_COMMIT
    + "/notebooks/DFU_PHASE3_XAI_GPU_RESUME_V3_OneCell.ipynb"
)

print("=" * 100)
print(VERSION)
print("VALID JSON LOADER | GPU-AWARE | READ-ONLY | RESUME-SAFE")
print("Primary training: FORBIDDEN")
print("Primary 45-trial modification: FORBIDDEN")
print("All Phase-2/Phase-3 tables and plots display in THIS SAME OUTPUT CELL.")
print("=" * 100)

raw_text = urllib.request.urlopen(SOURCE_CONTAINER_URL, timeout=120).read().decode("utf-8", errors="strict")
marker = "PAYLOAD = " + "\'\'\'"
start = raw_text.find(marker)
if start < 0:
    raise RuntimeError("Pinned V3 source container does not contain the expected PAYLOAD marker")
start += len(marker)
end = raw_text.find("\'\'\'", start)
if end < 0:
    raise RuntimeError("Pinned V3 source container PAYLOAD terminator not found")
payload = "".join(raw_text[start:end].split())
try:
    source = zlib.decompress(base64.b64decode(payload, validate=True))
except Exception as exc:
    raise RuntimeError(f"Pinned V3 payload decode failed: {type(exc).__name__}: {exc}") from exc
actual_sha256 = hashlib.sha256(source).hexdigest()
if actual_sha256 != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "Pinned V3 decoded-source SHA256 mismatch: "
        f"expected={EXPECTED_SOURCE_SHA256} actual={actual_sha256}"
    )
print("Decoded V3 source SHA256: PASS", actual_sha256)
text = source.decode("utf-8")
compile(text, "dfu_phase3_xai_gpu_resume_v3.py", "exec")
print("Decoded V3 source compile: PASS")
print("Starting GPU-aware resume-safe XAI...")
exec(compile(text, "dfu_phase3_xai_gpu_resume_v3.py", "exec"), globals())
